# Uber Ride Cancellation Prediction

## Objective
Predict whether a ride booking will be cancelled by the customer
or completed, helping Uber drivers anticipate potential wasted trips.

## Target
- 0: Completed
- 1: Cancelled by Customer

## Features
- Vehicle Type
- Pickup Location
- Drop Location
- Hour
- DayOfWeek
- Month
- Avg VTAT

## Models
- Logistic Regression
- Random Forest

## Evaluation
- Validation average precision for model selection
- Validation F1-score for threshold selection
- Final test evaluation using the selected model and threshold

## Data leakage precautions
Payment Method, Ride Distance, and Booking Value are excluded
because their missingness perfectly identifies customer cancellations
in this dataset.

In [49]:
import importlib
import src.preproc

importlib.reload(src.preproc)

from src.preproc import prepare_data, create_preprocessor

In [50]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from src.preproc import prepare_data, create_preprocessor

In [ ]:
df = pd.read_csv("../data/uberData.csv")

expanded_features = [
    "Vehicle Type",
    "Pickup Location",
    "Drop Location",
    "Hour",
    "DayOfWeek",
    "Month",
    "Avg VTAT"
]

X, y = prepare_data(df, features=expanded_features)

print("Dataset shape:", X.shape)
print("\nTarget distribution:")
print(y.value_counts())
print("\nTarget proportions:")
print(y.value_counts(normalize=True).round(4))
print("\nMissing values in selected features:")
print(X.isna().sum())

Dataset shape: (103500, 7)

Target distribution:
target
0    93000
1    10500
Name: count, dtype: int64

Target proportions:
target
0    0.8986
1    0.1014
Name: proportion, dtype: float64


In [52]:
# First, set aside 30% for validation and testing
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Split the remaining 30% equally into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

print("\nCancellation rates:")
print("Train:", round(y_train.mean(), 4))
print("Validation:", round(y_val.mean(), 4))
print("Test:", round(y_test.mean(), 4))

Training set: (72450, 7)
Validation set: (15525, 7)
Test set: (15525, 7)

Cancellation rates:
Train: 0.1014
Validation: 0.1014
Test: 0.1014


In [53]:
lr_model = Pipeline([
    ("preprocessor", create_preprocessor(expanded_features)),
    ("classifier", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

rf_model = Pipeline([
    ("preprocessor", create_preprocessor(expanded_features)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

lr_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

print("Both models trained successfully.")

Both models trained successfully.


In [54]:
lr_val_probs = lr_model.predict_proba(X_val)[:, 1]
rf_val_probs = rf_model.predict_proba(X_val)[:, 1]

lr_val_pred = (lr_val_probs >= 0.5).astype(int)
rf_val_pred = (rf_val_probs >= 0.5).astype(int)

print("LOGISTIC REGRESSION — VALIDATION")
print(classification_report(
    y_val, lr_val_pred, zero_division=0
))
print(
    "Average Precision:",
    round(average_precision_score(y_val, lr_val_probs), 4)
)

print("\nRANDOM FOREST — VALIDATION")
print(classification_report(
    y_val, rf_val_pred, zero_division=0
))
print(
    "Average Precision:",
    round(average_precision_score(y_val, rf_val_probs), 4)
)

LOGISTIC REGRESSION — VALIDATION
              precision    recall  f1-score   support

           0       0.94      0.66      0.77     13950
           1       0.17      0.62      0.27      1575

    accuracy                           0.66     15525
   macro avg       0.55      0.64      0.52     15525
weighted avg       0.86      0.66      0.72     15525

Average Precision: 0.4397

RANDOM FOREST — VALIDATION
              precision    recall  f1-score   support

           0       0.93      1.00      0.96     13950
           1       0.95      0.33      0.49      1575

    accuracy                           0.93     15525
   macro avg       0.94      0.66      0.73     15525
weighted avg       0.93      0.93      0.91     15525

Average Precision: 0.4766


In [55]:
lr_ap = average_precision_score(y_val, lr_val_probs)
rf_ap = average_precision_score(y_val, rf_val_probs)

if lr_ap >= rf_ap:
    best_model = lr_model
    best_model_name = "Logistic Regression"
    best_val_probs = lr_val_probs
else:
    best_model = rf_model
    best_model_name = "Random Forest"
    best_val_probs = rf_val_probs

print("Model selected for threshold exploration:", best_model_name)
print(f"Validation Average Precision: {max(lr_ap, rf_ap):.4f}")

threshold_results = []

for threshold in np.arange(0.05, 0.96, 0.05):
    predictions = (best_val_probs >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print("\nThreshold comparison:")
print(threshold_df.round(3).to_string(index=False))

Model selected for threshold exploration:

 Random Forest
Validation Average Precision: 0.4766

Threshold comparison:
 Threshold  Precision  Recall    F1
      0.05      0.124   0.990 0.221
      0.10      0.135   0.909 0.235
      0.15      0.162   0.721 0.265
      0.20      0.199   0.589 0.298
      0.25      0.275   0.452 0.342
      0.30      0.421   0.385 0.402
      0.35      0.656   0.349 0.455
      0.40      0.775   0.336 0.469
      0.45      0.896   0.333 0.485
      0.50      0.947   0.331 0.491
      0.55      0.977   0.330 0.493
      0.60      0.990   0.330 0.495
      0.65      0.994   0.330 0.495
      0.70      0.994   0.330 0.495
      0.75      0.998   0.329 0.495
      0.80      1.000   0.327 0.493
      0.85      1.000   0.314 0.478
      0.90      1.000   0.286 0.444
      0.95      1.000   0.206 0.342


In [56]:
best_row_index = threshold_df["F1"].idxmax()

best_threshold = threshold_df.loc[
    best_row_index, "Threshold"
]

print("Selected model:", best_model_name)
print("Selected threshold:", best_threshold)
print(
    "Validation F1-score:",
    round(threshold_df.loc[best_row_index, "F1"], 4)
)

Selected model: Random Forest
Selected threshold: 0.65
Validation F1-score: 0.495


In [57]:
test_probs = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_probs >= best_threshold).astype(int)

print("FINAL TEST RESULTS")
print("Selected model:", best_model_name)
print("Selected threshold:", best_threshold)

print("\nClassification Report:")
print(classification_report(
    y_test,
    test_pred,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print(
    "\nTest Average Precision:",
    round(average_precision_score(y_test, test_probs), 4)
)

print(
    "Actual customer cancellations:",
    int(y_test.sum())
)

print(
    "Predicted customer cancellations:",
    int(test_pred.sum())
)

FINAL TEST RESULTS
Selected model: Random Forest
Selected threshold: 0.65

Classification Report:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96     13950
           1       1.00      0.35      0.52      1575

    accuracy                           0.93     15525
   macro avg       0.97      0.67      0.74     15525
weighted avg       0.94      0.93      0.92     15525

Confusion Matrix:
[[13950     0]
 [ 1026   549]]

Test Average Precision: 0.4894
Actual customer cancellations: 1575
Predicted customer cancellations: 549
